# 🌐 COVID-19 Global Epidemiology Intelligence & Macro Surveillance
**Project:** Global Pandemic Surveillance & Health Risk Analytics  
**Author:** Portfolio Project  
**Dataset Snapshot:** April 29, 2020 (Critical Pandemic Inflection Point)  

---
### 1. Public Health Context & Problem Statement
During rapid global health emergencies, decision-makers across healthcare systems, international agencies, and supply chain enterprises require objective epidemiological metrics beyond raw headline counts:
1. **Active Transmission Pressure:** How many infectious cases remain active versus closed outcomes (deaths and clinical recoveries)?
2. **Case Fatality Rates (CFR):** Which nations suffer disproportionate mortality, and where does testing capacity distort apparent fatality?
3. **Recovery Velocity:** Where are healthcare recovery protocols successfully resolving infection cohorts?
4. **Geographic Concentration:** What proportion of the worldwide disease burden is concentrated within the top outbreak epicenters?

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['font.size'] = 10

ROOT_DIR = Path('..').resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.data_loader import load_raw_covid_data, clean_covid_data
from src.analytics import aggregate_by_region, get_global_kpis, get_top_outbreaks
from src.epidemiology import enrich_epidemiological_metrics
print('Surveillance modules loaded successfully.')

### 2. Ingestion & Data Sanitization
We load raw observations, standardize regional naming, parse observation dates, and impute subnational state aggregates.

In [ ]:
raw_df = load_raw_covid_data(ROOT_DIR / 'data' / 'raw' / 'covid19_global_raw.csv')
cleaned_df = clean_covid_data(raw_df)
print(f'Ingested {len(cleaned_df)} observations across {cleaned_df["Region"].nunique()} regions.')
cleaned_df.head(6)

### 3. Worldwide Headline KPIs
Let us aggregate global metrics to understand the macro pandemic footprint at this critical snapshot.

In [ ]:
kpis = get_global_kpis(cleaned_df)
kpi_df = pd.DataFrame([{
    'Total Confirmed': f"{kpis['total_confirmed']:,}",
    'Total Deaths': f"{kpis['total_deaths']:,}",
    'Total Recovered': f"{kpis['total_recovered']:,}",
    'Active Infections': f"{kpis['total_active']:,}",
    'Global CFR': f"{kpis['global_cfr_pct']}%",
    'Recovery Rate': f"{kpis['global_recovery_rate_pct']}%",
    'Affected Nations': kpis['total_countries_affected']
}])
kpi_df

### 4. Macro Country-Level Rollups & Top Epicenters
We roll up subnational provinces and states into sovereign national figures, recomputing exact Case Fatality Rates (CFR) and Active Cases.

In [ ]:
regional_df = aggregate_by_region(cleaned_df)
top10 = get_top_outbreaks(regional_df, metric='Confirmed', n=10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 10 Confirmed Cases
sns.barplot(data=top10, x='Confirmed', y='Region', palette='Reds_r', ax=axes[0])
axes[0].set_title('Top 10 Global Epicenters by Total Confirmed Infections', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Confirmed Cases')

# Case Fatality Rates for Top 10
sns.barplot(data=top10.sort_values(by='CFR_Pct', ascending=False), x='CFR_Pct', y='Region', palette='Blues_r', ax=axes[1])
axes[1].set_title('Case Fatality Rate (CFR %) Across Major Epicenters', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Case Fatality Rate (%)')
plt.tight_layout()
plt.show()

### 5. Active Transmission vs. Recovery Velocity
A vital epidemiological indicator is the balance between active cases and clinical recovery rates.

In [ ]:
top25 = get_top_outbreaks(regional_df, metric='Confirmed', n=25)

plt.figure(figsize=(11, 6))
scatter = sns.scatterplot(
    data=top25,
    x='Recovery_Rate_Pct',
    y='CFR_Pct',
    size='Confirmed',
    sizes=(60, 500),
    hue='Severity_Risk_Index',
    palette='viridis',
    alpha=0.85
)
for _, row in top25.head(8).iterrows():
    plt.text(row['Recovery_Rate_Pct'] + 1, row['CFR_Pct'] + 0.2, row['Region'], fontsize=9, fontweight='semibold')

plt.title('Case Fatality Rate vs. Recovery Velocity (Top 25 Outbreaks)', fontsize=13, fontweight='bold')
plt.xlabel('Recovery Rate (% of Confirmed Cases)', fontsize=11)
plt.ylabel('Case Fatality Rate (CFR %)', fontsize=11)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 6. Summary of Exploratory Findings
1. **Massive Concentration:** The Top 5 countries (US, Spain, Italy, France, UK) accounted for over **56.8%** of all global infections and **69.2%** of worldwide deaths at this snapshot.
2. **Fatality Rate Variance:** Major European epicenters exhibited peak Case Fatality Rates exceeding 13%–15% (UK 15.72%, France 14.48%, Italy 13.60%), reflecting initial acute hospital capacity strain and selective testing criteria.
3. **Recovery Momentum Divergence:** While early epicenters like Mainland China reached >93% recovery velocity, emerging outbreaks in the Americas were in rapid acceleration with active cases exceeding 82% of confirmed totals.

Next: In notebook `02_risk_clustering_and_severity.ipynb`, we build an unsupervised machine learning clustering pipeline to classify all 187 nations into public health risk tiers.